<a href="https://colab.research.google.com/github/Fantiflex/Cognitive_biaises/blob/main/02_generate_social_matricesv2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Purpose : Generate ONE shared set of stimuli for the social perception task.**
Produces two stimulus types from a single set of design parameters:


1.   SOCIAL matrices   — CFD face grids (Black / White / Men / Women)
2.   NON-SOCIAL matrices — dark-grey / light-grey filled circle grids used as a control condition, matched to critical trial prevalences

**Output structure** :
           

**Naming convention**
T<NNN>_<category>_<pct>pct.jpg
NNN      = zero-padded trial index (within stimulus type)
category = black | white | men | women | dark | light
pct      = true minority % (10 / 20 / 30 / 40 / 50)




**Design**



*Social:*     4 categories × 5 prevalence levels × N_REPS reps
* Critical : Black, White
* Filler   : Men, Women
*Non-social*: 2 categories × 5 prevalence levels × N_REPS reps
Control  : Dark (minority), Light (majority)
* same prevalence pool as social critical trials —
* dark circle ≡ minority face (Black); mirrors logic —

#           To scale up/down: ONLY edit N_REPS (and circle aesthetics if needed).
All counts, filenames, and the manifest update automatically.

In [1]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)
import sys
import cv2
import numpy as np
from PIL import Image

# Vous pouvez maintenant relancer votre boucle sur 'minority_imgs' ou 'prep_face'


Mounted at /content/drive


In [2]:
from pathlib import Path

base_dir = Path("/content/drive/MyDrive/Colab_notebooks/Ulm")

print("base_dir exists:", base_dir.exists())
print("Contenu de base_dir:")
for p in base_dir.iterdir():
    print(p.name)

base_dir exists: True
Contenu de base_dir:
Images
CFD_codebook.xlsx
Untitled0.ipynb
outputs
minority_Faces
majority_Faces
matrices
03_generate_nonsocial_matrices.ipynb
__pycache__
01_select_cfd_images.ipynb
Image_processing.ipynb
02_generate_social_matrices.ipynb


**Imports**

In [3]:
from pathlib import Path
import random
import shutil
import pandas as pd
import numpy as np
from PIL import Image, ImageDraw

**Files loading & cleaning**

In [4]:
base_dir = Path("/content/drive/MyDrive/Colab_notebooks/Ulm")



STIM_ROOT = base_dir

OUT_MIN = STIM_ROOT /"minority_Faces"
OUT_MAJ = STIM_ROOT /"majority_Faces"

SOCIAL_DIR = STIM_ROOT / "matrices" / "social"
NONSOCIAL_DIR = STIM_ROOT / "matrices" / "nonsocial"


SOCIAL_DIR.mkdir(parents=True, exist_ok=True)
NONSOCIAL_DIR.mkdir(parents=True, exist_ok=True)

print("OUT_MIN exists:", OUT_MIN.exists())
print("OUT_MAJ exists:", OUT_MAJ.exists())

print("Images minority:", len(list(OUT_MIN.rglob("*"))))
print("Images majority:", len(list(OUT_MAJ.rglob("*"))))

# Nettoyer les anciennes matrices si on relance le script
for folder in [SOCIAL_DIR, NONSOCIAL_DIR]:
    for old_file in folder.glob("*.jpg"):
        old_file.unlink()

OUT_MIN exists: True
OUT_MAJ exists: True
Images minority: 122
Images majority: 122


**Settings matrices**

In [5]:
GRID_COLS = 10
GRID_ROWS = 10
CELL_PX = 100
JPEG_QUAL = 82

PREVALENCE_LEVELS = [10, 20, 30, 40, 50]
N_REPS = 2
################# À ADAPTER AVEC LE CONTRASTE DE LA MATRICE SELECTIONNÉE ##############
CIRCLE_DARK_COL = "grey30"
CIRCLE_LIGHT_COL = "grey75"
CIRCLE_BG_COL = "white"
CIRCLE_RADIUS_PCT = 0.42

GLOBAL_SEED = 2022

n_faces = GRID_COLS * GRID_ROWS

N_SOCIAL_CATEGORIES = 4
N_NONSOCIAL_CATS = 2

N_TOTAL_SOCIAL = N_SOCIAL_CATEGORIES * len(PREVALENCE_LEVELS) * N_REPS
N_TOTAL_NONSOCIAL = N_NONSOCIAL_CATS * len(PREVALENCE_LEVELS) * N_REPS

# **Load faces images**
éxécution environ 1min30

In [6]:

IMAGE_EXTENSIONS = {".jpg"}

def load_paths(folder, label):
    paths = [
        p for p in folder.rglob("*")
        if p.is_file()
        and not p.name.startswith(".")
        and p.suffix.lower() in IMAGE_EXTENSIONS
    ]

    if len(paths) == 0:
        raise FileNotFoundError(
            f"No images found in {folder}. "
            "Run the previous image selection/copy step first."
        )

    print(f"{label}: {len(paths)} images")
    return paths


minority_paths = load_paths(OUT_MIN, "Minority / Black")
majority_paths = load_paths(OUT_MAJ, "Majority / White")


def prep_face(path):
    img = cv2.imread(path)

    height, width, _ = img.shape
    side = min(width, height)

    left = (width - side) // 2
    top = (height - side) // 2
    right = left + side
    bottom = top + side


    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_pil = Image.fromarray(img_rgb)
    img_pil = img_pil.resize((CELL_PX, CELL_PX))
    # On retourne l'image au format PIL pour que le .paste() fonctionne à l'étape suivante
    return img, img_pil, side



minority_imgs = [prep_face(p) for p in minority_paths]
majority_imgs = [prep_face(p) for p in majority_paths]

print(f"{len(minority_imgs)} minority + {len(majority_imgs)} majority faces loaded")

Minority / Black: 122 images
Majority / White: 122 images
122 minority + 122 majority faces loaded


**Select faces w criteria**

In [7]:
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)
############## SOCIAL TRIAL ################
social_trials = []

for question_category in ["Black", "White"]:
    for true_minority_pct in PREVALENCE_LEVELS:
        for rep in range(1, N_REPS + 1):
            social_trials.append({
                "question_category": question_category,
                "true_minority_pct": true_minority_pct,
                "stim_type": "social",
                "trial_type": "critical" if question_category in ["Black", "White"] else "filler",
                "true_majority_pct": 100 - true_minority_pct
            })

social_trials = pd.DataFrame(social_trials)
social_trials = social_trials.sample(frac=1, random_state=GLOBAL_SEED).reset_index(drop=True)
social_trials["trial_idx"] = np.arange(1, len(social_trials) + 1)



**CREATE FACE RECORDS WITH INDIVIDUAL LUMINANCE (NE PAS RELANCER)**

Éxécution environ 3 to 5min

In [18]:
%run "/content/drive/MyDrive/Colab_notebooks/Ulm/Image_processing.ipynb"

def prep_face_record(path, group):
    """
    Prépare une image visage + garde ses métadonnées.
    group = "minority" ou "majority"
    """
    img,img_pil,_= prep_face(path)
    lum = extract_luminance_hrnet(img)

    return {
        "path": str(path),
        "filename": Path(path).name,
        "group": group,
        "image": img_pil,
        "luminance": lum
    }


minority_records = [
    prep_face_record(p, "minority")
    for p in minority_paths
]


majority_records = [
    prep_face_record(p, "majority")
    for p in majority_paths
]

print("minority_records:", len(minority_records))
print("majority_records:", len(majority_records))

Chargement du modèle HRNet...
minority_records: 122
majority_records: 122


**Functions to ASSEMBLE MATRICES**

In [30]:
def assemble_grid(cells):
    matrix = Image.new("RGB", (GRID_COLS * CELL_PX, GRID_ROWS * CELL_PX))

    for idx, cell in enumerate(cells):
        row = idx // GRID_COLS
        col = idx % GRID_COLS

        x = col * CELL_PX
        y = row * CELL_PX

        matrix.paste(cell, (x, y))

    return matrix



def make_face_matrix_with_cell_info(n_minority, img_seed):
    rng = random.Random(img_seed)
    n_majority=n_faces-n_minority
    selected_minority = rng.sample(minority_records, n_minority)
    selected_majority = rng.sample(majority_records, n_majority)
    selected = selected_minority + selected_majority
    rng.shuffle(selected)

    cells = [rec["image"] for rec in selected]

    n_majority = n_faces - n_minority
    matrix = assemble_grid(cells)

    cell_info = []

    for idx, rec in enumerate(selected):
        row = idx // GRID_COLS
        col = idx % GRID_COLS

        cell_info.append({
            "cell_idx": idx,
            "row": row,
            "col": col,
            "group": rec["group"],
            "source_face": rec["path"],
            "source_filename": rec["filename"],
            "face_luminance": rec["luminance"]
        })

    return matrix, pd.DataFrame(cell_info),


# **Generate the actual visual matrices**

In [31]:
social_manifest = []

print(f"Generating {N_TOTAL_SOCIAL} social matrices...")
cell_info_rows = []


for i, row in social_trials.iterrows():
    trial_idx = int(row["trial_idx"])
    category = row["question_category"]
    true_minority_pct = int(row["true_minority_pct"])

    fname = f"T{trial_idx:03d}_{category.lower()}_{true_minority_pct}pct.jpg"
    fpath = SOCIAL_DIR / fname

    img_seed = GLOBAL_SEED + trial_idx * 13
    n_min = round(n_faces * true_minority_pct / 100)

    n_majority=n_faces-n_min
    img, cell_info = make_face_matrix_with_cell_info(n_min, img_seed)

    img.save(fpath, format="JPEG", quality=JPEG_QUAL)

    cell_info["trial_idx"] = trial_idx
    cell_info["social_filename"] = fname
    cell_info["true_minority_pct"] = true_minority_pct
    cell_info["category"] = category
    cell_info["img_seed"] = img_seed

    cell_info_rows.append(cell_info)

    social_manifest.append({
        "stim_type": "social",
        "trial_idx": trial_idx,
        "filename": fname,
        "url_stub": f"social/{fname}",
        "trial_type": row["trial_type"],
        "category": category,
        "true_minority_pct": true_minority_pct,
        "true_majority_pct": int(row["true_majority_pct"]),
        "img_seed": img_seed
    })

    if (i + 1) % 10 == 0:
        print(f"{i + 1}/{N_TOTAL_SOCIAL} done")


social_cell_manifest = pd.concat(cell_info_rows, ignore_index=True)

social_cell_manifest_path = base_dir / "matrices" / "social_cell_manifest.csv"
social_cell_manifest.to_csv(social_cell_manifest_path, index=False)

print("Cell-level social manifest saved:", social_cell_manifest_path)
display(social_cell_manifest.head())

Generating 40 social matrices...
10/40 done
20/40 done
Cell-level social manifest saved: /content/drive/MyDrive/Colab_notebooks/Ulm/matrices/social_cell_manifest.csv


,cell_idx,row,col,group,source_face,source_filename,face_luminance,trial_idx,social_filename,true_minority_pct,category,img_seed
0,0,0,0,minority,/content/drive/MyDrive/Colab_notebooks/Ulm/min...,BM-223_neutral.jpg,77.864641,1,T001_white_50pct.jpg,50,White,2035
1,1,0,1,minority,/content/drive/MyDrive/Colab_notebooks/Ulm/min...,BM-025_neutral.jpg,88.756903,1,T001_white_50pct.jpg,50,White,2035
2,2,0,2,majority,/content/drive/MyDrive/Colab_notebooks/Ulm/maj...,WF-020_neutral.jpg,162.372469,1,T001_white_50pct.jpg,50,White,2035
3,3,0,3,minority,/content/drive/MyDrive/Colab_notebooks/Ulm/min...,BF-241_neutral.jpg,131.020780,1,T001_white_50pct.jpg,50,White,2035
4,4,0,4,minority,/content/drive/MyDrive/Colab_notebooks/Ulm/min...,BF-002_neutral.jpg,110.653980,1,T001_white_50pct.jpg,50,White,2035


In [17]:
if len(minority_records) > 0:
        premier_element = minority_records[0]
        print("Clés disponibles dans votre dictionnaire :", list(premier_element.keys()))

Clés disponibles dans votre dictionnaire : ['path', 'filename', 'group', 'image', 'luminance']
